---
# **Cuaderno 3:**  Analisis de datos de ORPHANET
---
* Author: Melissa Ormeño; Alexis Salas
* Laboratory: Cancer Molecular Dynamics
* Date: Agosto 2025

**Aims:**

* Analiza los datos obtenidos al filtrar por gen un archivo de mutaciones de la base de datos del ICGC

In [1]:
import pandas as pd

In [2]:
chunksize=200_000

df = pd.read_csv("../data/ClinVar/clinvar.vcf.gz", sep="\t",comment="#", low_memory=False)


In [3]:
df

,1,66926,3385321,AG,A,.,..1,"ALLELEID=3544463;CLNDISDB=Human_Phenotype_Ontology:HP:0000547,MONDO:MONDO:0019200,MeSH:D012174,MedGen:C0035334,OMIM:268000,OMIM:PS268000,Orphanet:791;CLNDN=Retinitis_pigmentosa;CLNHGVS=NC_000001.11:g.66927del;CLNREVSTAT=criteria_provided,_single_submitter;CLNSIG=Uncertain_significance;CLNSIGSCV=SCV005419006;CLNVC=Deletion;CLNVCSO=SO:0000159;GENEINFO=OR4F5:79501;MC=SO:0001627|intron_variant;ORIGIN=0"
0,1,69134,2205837,A,G,.,.,ALLELEID=2193183;CLNDISDB=MedGen:CN169374;CLND...
1,1,69308,3925305,A,G,.,.,ALLELEID=4039319;CLNDISDB=MedGen:CN169374;CLND...
2,1,69314,3205580,T,G,.,.,ALLELEID=3374047;CLNDISDB=MedGen:CN169374;CLND...
3,1,69404,3925306,T,C,.,.,ALLELEID=4039320;CLNDISDB=MedGen:CN169374;CLND...
4,1,69423,3205581,G,A,.,.,ALLELEID=3374048;CLNDISDB=MedGen:CN169374;CLND...
...,...,...,...,...,...,...,...,...
3677944,NT_187693.1,273866,2237818,A,C,.,.,ALLELEID=2232003;CLNDISDB=MedGen:CN169374;CLND...
3677945,NT_187693.1,274185,3778023,C,T,.,.,ALLELEID=3894028;CLNDISDB=MedGen:C3661900;CLND...
3677946,NT_187693.1,274366,2206666,G,C,.,.,ALLELEID=2200058;CLNDISDB=MedGen:CN169374;CLND...
3677947,NT_187693.1,275068,2241971,T,C,.,.,ALLELEID=2226217;CLNDISDB=MedGen:CN169374;CLND...


In [4]:
import pandas as pd

vcf_path = "../data/ClinVar/clinvar.vcf.gz"

cols = ["#CHROM","POS","ID","REF","ALT","QUAL","FILTER","INFO"]  # VCF sin muestras
df = pd.read_csv(
    vcf_path,
    compression="gzip",
    sep="\t",
    comment="#",
    names=cols,
    usecols=cols,           # evita leer columnas de muestras si existieran
    dtype={"#CHROM":"string","POS":"int64","ID":"string",
           "REF":"string","ALT":"string","QUAL":"string","FILTER":"string","INFO":"string"}
)
df.head()


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO
0,1,66926,3385321,AG,A,.,.,ALLELEID=3544463;CLNDISDB=Human_Phenotype_Onto...
1,1,69134,2205837,A,G,.,.,ALLELEID=2193183;CLNDISDB=MedGen:CN169374;CLND...
2,1,69308,3925305,A,G,.,.,ALLELEID=4039319;CLNDISDB=MedGen:CN169374;CLND...
3,1,69314,3205580,T,G,.,.,ALLELEID=3374047;CLNDISDB=MedGen:CN169374;CLND...
4,1,69404,3925306,T,C,.,.,ALLELEID=4039320;CLNDISDB=MedGen:CN169374;CLND...


In [5]:
import pandas as pd

# Si ya leíste el VCF y te quedaste con INFO:
# df tiene una columna "INFO" de tipo string
def info_to_wide(info_series):
    dicts = info_series.fillna("").str.split(";").apply(
        lambda parts: {k: v for k, v in
                       (p.split("=", 1) for p in parts if "=" in p)}
    )
    return pd.json_normalize(dicts)  # columnas = claves, filas = variantes

info_df = info_to_wide(df["INFO"])
# Si quieres SOLO las columnas de INFO:
df = info_df


In [15]:
df

,ALLELEID,CLNDISDB,CLNDN,CLNHGVS,CLNREVSTAT,CLNSIG,CLNSIGSCV,CLNVC,CLNVCSO,GENEINFO,...,ONCDISDB,ONCDN,ONCREVSTAT,ONCSCV,SCI,SCIREVSTAT,DBVARID,SCIDISDB,SCIDN,SCISCV
0,3544463,"Human_Phenotype_Ontology:HP:0000547,MONDO:MOND...",Retinitis_pigmentosa,NC_000001.11:g.66927del,"criteria_provided,_single_submitter",Uncertain_significance,SCV005419006,Deletion,SO:0000159,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2193183,MedGen:CN169374,not_specified,NC_000001.11:g.69134A>G,"criteria_provided,_single_submitter",Likely_benign,SCV003526545,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4039319,MedGen:CN169374,not_specified,NC_000001.11:g.69308A>G,"criteria_provided,_single_submitter",Uncertain_significance,SCV006120601,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3374047,MedGen:CN169374,not_specified,NC_000001.11:g.69314T>G,"criteria_provided,_single_submitter",Uncertain_significance,SCV004995495,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4039320,MedGen:CN169374,not_specified,NC_000001.11:g.69404T>C,"criteria_provided,_single_submitter",Uncertain_significance,SCV006120602,single_nucleotide_variant,SO:0001483,OR4F5:79501,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3677945,2232003,MedGen:CN169374,not_specified,NaN,"criteria_provided,_single_submitter",Uncertain_significance,SCV003560205,single_nucleotide_variant,SO:0001483,LILRA3:11026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677946,3894028,MedGen:C3661900,not_provided,NaN,"criteria_provided,_single_submitter",Likely_benign,SCV005911038,single_nucleotide_variant,SO:0001483,LILRA3:11026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677947,2200058,MedGen:CN169374,not_specified,NaN,"criteria_provided,_single_submitter",Uncertain_significance,SCV003533488,single_nucleotide_variant,SO:0001483,LILRA3:11026,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3677948,2226217,MedGen:CN169374,not_specified,NaN,"criteria_provided,_single_submitter",Uncertain_significance,SCV003570821,single_nucleotide_variant,SO:0001483,LILRA3:11026|LOC126057115:126057115,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
df = df.dropna(subset = "CLNSIG", axis=1)
df_loc = df.loc[df["CLNSIG"].str.contains("Uncertain_significance")] 

KeyError: ['CLNSIG']